In [5]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [6]:
from langchain_core.documents import Document

# sample_doc = Document(

#     page_content="Hello world",
#     metadata={"source": "https://www.google.com"}
# )

In [7]:
# type(sample_doc)

In [8]:
# Text Data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("Python.txt", encoding="utf-8")

C:\Users\meshi\AppData\Local\Temp\ipykernel_15868\1792886827.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [9]:
document = loader.load()

# INgestion Pipeline

In [10]:
#DAta => Documents

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [11]:
def load_all_pdfs():
    folder_path = "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            #complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1


    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [12]:
all_pdf_documents = load_all_pdfs()

total pdfs: 3
total pages: 122


In [13]:
#chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_doc = text_splitter.split_documents(documents)
    return chunked_doc

In [14]:
chunks = split_docs(all_pdf_documents)

In [15]:
len(chunks)

942

### Embedding

In [16]:
from sentence_transformers import SentenceTransformer

In [17]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model=SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=False)
        print("embeddings shape:", embeddings.shape)
        return embeddings
        

In [18]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


In [19]:
import chromadb
import uuid
import os


class VectorStoreManager:
    def __init__(
        self,
        persist_directory="data/vector_store",
        collection_name="pdf_documents"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # Create a client
        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        # Create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "vector store collection for pdf embeddings in RAG"
            }
        )

        print(
            "initialized the vector store with collection:",
            self.collection_name
        )

        print(
            "docs in collection:",
            self.collection.count()
        )

    def add_documents(self, documents, embeddings):

        if len(documents) != len(embeddings):
            raise ValueError(
                "num of documents does not match num of embeddings"
            )

        # Store => IDs, embeddings, documents, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for doc, embedding in zip(documents, embeddings):

            # Unique ID
            doc_id = f"doc_{uuid.uuid4()}"

            ids.append(doc_id)

            # Document text
            documents_content.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

            # Metadata
            metadata = doc.metadata

            # Chroma metadata must contain simple values
            all_metadata.append(metadata)

        # Add everything to Chroma
        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            documents=documents_content,
            metadatas=all_metadata
        )

        print(
            f"Added {len(documents)} documents to vector store"
        )

        print(
            "docs in collection:",
            self.collection.count()
        )



In [20]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 2826


In [21]:
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

embeddings shape: (942, 384)
Added 942 documents to vector store
docs in collection: 3768


In [24]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [25]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [26]:
rag_retriever.retrieve("What is encoder decoder")

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_d53378b3-7b91-4c59-a5d7-8e842ab1857b',
  'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention',
  'metadata': {'producer': 'pdfTeX-1.40.25',
   'total_pages': 15,
   'moddate': '2024-04-10T21:11:43+00:00',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'creator': 'LaTeX with hyperref',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'trapped': '/False',
   'keywords': '',
   'source': 'data\\research1.pdf',
   'page': 2,
   'page_label': '3',
   'author': '',
   'title': '',
   'subject': ''},
  'di

In [27]:
API_KEY_OPENAI = "ENTER YOUR OPEN API KEY HERE" #it has to be in "here"

In [28]:
# !pip install langchain-openai

In [29]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [30]:
#generate our Retrieval- Augmented Output

In [37]:
def generate_output(query, rag_retriever, llm, top_k=3):
    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    #context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query}"""
    response = llm.invoke(prompt)
    return response.content

In [39]:
answer = generate_output("what is RAG?", rag_retriever, llm)

embeddings shape: (1, 384)
retrieved 3 documents


OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
print(answer)